# Tutorial: Planner Introspection and Plan Quality Lab

Audience:
- Agent architects tuning planner protocols and decomposition policies.
- Data scientists designing plan-quality metrics.
- AI engineers validating plan format and dependency semantics.

Prerequisites:
- Notebook 1 completed.
- ROMA source importable from local environment.
- Optional: `OPENAI_API_KEY` for live planner inference.

Learning goals:
- Inspect exact planner contract and dependency format.
- Validate how dependency indices become TaskDAG edges.
- Run planner inference and compute quantitative plan-shape metrics.


## Outline

1. Setup.
2. Planner schema and dependency format.
3. Synthetic planner output -> subgraph conversion lab.
4. Live planner inference lab (optional API call).
5. Plan quality metrics and interpretation.
6. Pitfalls and extension experiments.


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "roma_dspy").exists():
    for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
        if (candidate / "src" / "roma_dspy").exists():
            REPO_ROOT = candidate
            break

SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("REPO_ROOT:", REPO_ROOT)
print("SRC_PATH added:", SRC_PATH)


REPO_ROOT: /Users/admin/TuanDung/repos/ROMA
SRC_PATH added: /Users/admin/TuanDung/repos/ROMA/src


In [2]:
import asyncio
import inspect
import json
import os
from dataclasses import asdict
from pathlib import Path
from types import SimpleNamespace

from roma_dspy.config.manager import ConfigManager
from roma_dspy.core.engine.dag import TaskDAG
from roma_dspy.core.engine.runtime import ModuleRuntime
from roma_dspy.core.factory.agent_factory import AgentFactory
from roma_dspy.core.registry.agent_registry import AgentRegistry
from roma_dspy.core.signatures.base_models.subtask import SubTask
from roma_dspy.core.signatures.base_models.task_node import TaskNode
from roma_dspy.core.signatures.signatures import PlannerSignature
from roma_dspy.types import AgentType, TaskType

cfg_manager = ConfigManager(config_dir=REPO_ROOT / "config")
cfg_path = REPO_ROOT / "config" / "local" / "openai_config.yaml"
if not cfg_path.exists():
    cfg_path = REPO_ROOT / "config" / "defaults" / "config.yaml"

config = cfg_manager.load_config(config_path=str(cfg_path))
registry = AgentRegistry()
registry.initialize_from_config(config, factory=AgentFactory())
runtime = ModuleRuntime(registry=registry, config=config)

print("Config loaded from:", cfg_path)
print("Planner model:", registry.get_agent(AgentType.PLANNER).get_model_config().get("model"))


2026-02-24 13:47:28.968 | INFO     | roma_dspy.core.predictors.code_act_patch:apply_code_act_patch:96 - Applied CodeAct patch to inject typing imports into interpreter
2026-02-24 13:47:28.985 | DEBUG    | roma_dspy.config.manager:load_config:74 - Loading config: path=/Users/admin/TuanDung/repos/ROMA/config/local/openai_config.yaml, profile=None, overrides=None, env_prefix=ROMA_
2026-02-24 13:47:28.986 | DEBUG    | roma_dspy.config.manager:load_config:83 - Initialized empty base config (defaults applied in validation)
2026-02-24 13:47:28.996 | DEBUG    | roma_dspy.config.manager:_load_yaml:149 - Loaded and cached config from /Users/admin/TuanDung/repos/ROMA/config/local/openai_config.yaml
2026-02-24 13:47:28.998 | DEBUG    | roma_dspy.config.manager:load_config:93 - Merged YAML config from /Users/admin/TuanDung/repos/ROMA/config/local/openai_config.yaml
2026-02-24 13:47:29.004 | DEBUG    | roma_dspy.config.manager:load_config:123 - Resolved interpolations
2026-02-24 13:47:29.005 | INFO 

Config loaded from: /Users/admin/TuanDung/repos/ROMA/config/local/openai_config.yaml
Planner model: openai/gpt-4o-mini


## Planner Output Contract

Planner returns:
- `subtasks: List[SubTask]`
- `dependencies_graph: Dict[str, List[str]]` where keys and values are **index strings**.

Then `ModuleRuntime._create_subtask_graph(...)` converts this index graph into task-id graph.


In [3]:
print("PlannerSignature source excerpt:")
print(chr(10).join(inspect.getsource(PlannerSignature).splitlines()[:50]))

print()
print("SubTask schema:")
print(json.dumps(SubTask.model_json_schema(), indent=2)[:1600])


PlannerSignature source excerpt:
class PlannerSignature(dspy.Signature):
    """
    Planner decomposition result.

    Contains the breakdown of a complex task into executable subtasks.
    """

    goal: str = dspy.InputField(
        description="Task that needs to be decomposed into subtasks through planner"
    )
    context: Optional[str] = dspy.InputField(
        default=None, description="Execution context (XML)"
    )
    subtasks: List[SubTask] = dspy.OutputField(
        description="List of generated subtasks from planner"
    )
    dependencies_graph: Optional[Dict[str, List[str]]] = dspy.OutputField(
        default=None,
        description="Task dependency mapping. Keys are subtask indices as strings (e.g., '0', '1'), values are lists of dependency indices as strings. Example: {'1': ['0'], '2': ['0', '1']}",
    )

SubTask schema:
{
  "$defs": {
    "TaskType": {
      "description": "MECE task classification for universal task decomposition.",
      "enum": [
        

## Synthetic Plan -> TaskDAG Conversion Lab

This lab isolates planner post-processing behavior without paying for an API call.


In [4]:
# Build a parent task and synthetic planner result
root_dag = TaskDAG(execution_id="planner-lab")
parent = TaskNode(
    goal="Analyze remote work impact on engineering productivity",
    execution_id=root_dag.execution_id,
    max_depth=4,
)
parent = root_dag.add_node(parent)

planner_result = SimpleNamespace(
    subtasks=[
        SubTask(goal="List productivity benefits", task_type=TaskType.RETRIEVE),
        SubTask(goal="List productivity risks", task_type=TaskType.RETRIEVE),
        SubTask(goal="Synthesize balanced recommendation", task_type=TaskType.THINK),
    ],
    dependencies_graph={
        "2": ["0", "1"],
    },
)

parent_after = runtime._create_subtask_graph(parent, root_dag, planner_result)
subgraph = root_dag.get_subgraph(parent_after.subgraph_id)

print("Parent subgraph_id:", parent_after.subgraph_id)
print("Subgraph task count:", len(subgraph.graph.nodes()))
print("Subgraph edge count:", len(subgraph.graph.edges()))

print()
print("Subgraph edges (dependency task_id -> task_id):")
for u, v in subgraph.graph.edges():
    print(u[:8], "->", v[:8])

print()
print("Task details:")
for task in subgraph.get_all_tasks(include_subgraphs=False):
    print({
        "task_id": task.task_id[:8],
        "goal": task.goal,
        "task_type": str(task.task_type),
        "dependencies": [d[:8] for d in task.dependencies],
    })


2026-02-24 13:47:29.108 | DEBUG    | roma_dspy.core.signatures.base_models.task_node:model_copy:69 - [TaskNode.model_copy] Preserving task_id: 9ac329b6...
2026-02-24 13:47:29.109 | DEBUG    | roma_dspy.core.signatures.base_models.task_node:model_copy:69 - [TaskNode.model_copy] Preserving task_id: a95eaad4...
2026-02-24 13:47:29.109 | DEBUG    | roma_dspy.core.engine.dag:_validate_dag_integrity:123 - DAG contains disconnected components (parallel tasks)
2026-02-24 13:47:29.110 | DEBUG    | roma_dspy.core.signatures.base_models.task_node:model_copy:69 - [TaskNode.model_copy] Preserving task_id: e6129a39...
2026-02-24 13:47:29.110 | DEBUG    | roma_dspy.core.engine.dag:_validate_dag_integrity:123 - DAG contains disconnected components (parallel tasks)
2026-02-24 13:47:29.110 | DEBUG    | roma_dspy.core.engine.dag:create_subgraph:340 - [create_subgraph] Subtask IDs in subgraph: ['a95eaad4', '9ac329b6', 'e6129a39']
2026-02-24 13:47:29.110 | DEBUG    | roma_dspy.core.engine.dag:create_subgra

Parent subgraph_id: bb8b3d92-81d8-4df0-9fa9-b99cf2267aa9_sub_67b03c39-7f93-426e-b3bd-c9a490f4be14
Subgraph task count: 3
Subgraph edge count: 2

Subgraph edges (dependency task_id -> task_id):
9ac329b6 -> e6129a39
a95eaad4 -> e6129a39

Task details:
{'task_id': '9ac329b6', 'goal': 'List productivity benefits', 'task_type': 'RETRIEVE', 'dependencies': []}
{'task_id': 'a95eaad4', 'goal': 'List productivity risks', 'task_type': 'RETRIEVE', 'dependencies': []}
{'task_id': 'e6129a39', 'goal': 'Synthesize balanced recommendation', 'task_type': 'THINK', 'dependencies': ['a95eaad4', '9ac329b6']}


## Live Planner Inference Lab (Optional)

Set `RUN_LIVE_PLANNER = True` to call the actual planner model.
This helps you inspect real decomposition behavior and validate policy assumptions.


In [5]:
RUN_LIVE_PLANNER = False  # Set True to run real planner inference
LIVE_GOAL = "Design a 2-week experiment to evaluate whether remote work changes engineering throughput and quality."

async def run_live_planner(goal: str):
    planner = registry.get_agent(AgentType.PLANNER)
    pred = await planner.aforward(goal=goal, context=None)
    return pred

if RUN_LIVE_PLANNER:
    if not os.getenv("OPENAI_API_KEY"):
        print("OPENAI_API_KEY not found; skipping live planner run.")
    else:
        live_result = asyncio.run(run_live_planner(LIVE_GOAL))
        print("Subtask count:", len(live_result.subtasks))
        print("Dependencies graph:", live_result.dependencies_graph)
        for idx, st in enumerate(live_result.subtasks):
            print(idx, {"goal": st.goal, "task_type": str(st.task_type), "dependencies": st.dependencies})
else:
    print("Live run skipped. Toggle RUN_LIVE_PLANNER=True to execute.")


Live run skipped. Toggle RUN_LIVE_PLANNER=True to execute.


## Plan Quality Metrics

A minimal metric set for planner diagnostics:
- `subtask_count`
- `edge_count`
- `dependency_density = edge_count / (n * (n - 1) / 2)`
- `max_in_degree` and `max_out_degree`

Use this as a baseline before introducing higher-level quality criteria.


In [6]:
def summarize_plan(subtasks, dependencies_graph):
    n = len(subtasks)
    edge_count = sum(len(v) for v in (dependencies_graph or {}).values())
    max_possible = (n * (n - 1)) / 2 if n > 1 else 1
    density = edge_count / max_possible if max_possible else 0.0

    in_degree = {i: 0 for i in range(n)}
    out_degree = {i: 0 for i in range(n)}

    for child_str, dep_list in (dependencies_graph or {}).items():
        child = int(child_str)
        in_degree[child] += len(dep_list)
        for dep_str in dep_list:
            out_degree[int(dep_str)] += 1

    return {
        "subtask_count": n,
        "edge_count": edge_count,
        "dependency_density": round(density, 4),
        "max_in_degree": max(in_degree.values()) if in_degree else 0,
        "max_out_degree": max(out_degree.values()) if out_degree else 0,
    }

synthetic_summary = summarize_plan(planner_result.subtasks, planner_result.dependencies_graph)
print("Synthetic plan metrics:", synthetic_summary)

if "live_result" in globals():
    print("Live plan metrics:", summarize_plan(live_result.subtasks, live_result.dependencies_graph))


Synthetic plan metrics: {'subtask_count': 3, 'edge_count': 2, 'dependency_density': 0.6667, 'max_in_degree': 2, 'max_out_degree': 1}


## Pitfalls And Extensions

Common pitfalls:
- Over-decomposition: too many shallow subtasks with weak dependency structure.
- Under-decomposition: one broad subtask hides unresolved reasoning complexity.
- Dependency misuse: dense all-to-all graphs reduce parallelism and increase latency.

Extensions:
1. Add task-type distribution metrics (`RETRIEVE/THINK/WRITE` balance).
2. Add planner consistency checks across repeated runs.
3. Add cost-aware scoring by combining plan complexity with token usage from execution history.
